In [11]:
from mistralai import Mistral
from typing import cast, List
from mistralai.models import MessagesTypedDict
import os
import base64
from pydantic import BaseModel
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()


# Define the Pydantic model
class CodeWilayah(BaseModel):
    no: int
    provinsi: str
    kabupaten_kota: str
    nama_kota: str
    singkatan_nama_kota: str
    parent_subdivision: str

class TableCodeWilayah(BaseModel):
    records: List[CodeWilayah]


api_key = os.getenv("MISTRAL_API_KEY")
mistral = Mistral(api_key=api_key)

In [12]:
current_path = os.getcwd()
script_dir = os.path.dirname(current_path)
input_dir = os.path.join(script_dir, "datas")
file_path = 'SNI_7657-2023\\SNI_7657-2023.pdf' # until 39.jpg
image_path = os.path.join(input_dir, file_path)

In [13]:
def encode_image(image_path):
    """Encode the image to base64."""
    try:
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')
    except FileNotFoundError:
        print(f"Error: The file {image_path} was not found.")
        return None
    except Exception as e:  # Added general exception handling
        print(f"Error: {e}")
        return None


In [14]:
base64_image = encode_image(image_path)

In [15]:
# Specify model
model = "pixtral-12b-latest"

In [16]:
messages = cast(List[MessagesTypedDict], [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "Extract All table rows from this image and return them as a list of structured records. The table contains multiple rows - extract every single row."
            },
            {
                "type": "image_url",
                "image_url": "https://akses-sni.bsn.go.id/dokumen/2023/SNI%207657-2023/files/page/13.jpg"
            },
            {
                "type": "image_url",
                "image_url": "https://akses-sni.bsn.go.id/dokumen/2023/SNI%207657-2023/files/page/14.jpg"
            }
        ]
    }
])

In [17]:
chat_response = mistral.chat.parse(
    model=model,
    messages=messages,
    temperature=0.1,
    response_format=TableCodeWilayah
)

In [19]:
try:
    # chat_response = mistral.chat.parse(
    #     model=model,
    #     messages=messages,
    #     temperature=0.1,  # Lower temperature for consistency
    #     response_format=TableCodeWilayah
    # )
    
    # Safe access to parsed data
    if (chat_response and 
        chat_response.choices and 
        len(chat_response.choices) > 0 and
        chat_response.choices[0].message and
        hasattr(chat_response.choices[0].message, 'parsed') and
        chat_response.choices[0].message.parsed):
        
        parsed_data = chat_response.choices[0].message.parsed
        
        if hasattr(parsed_data, 'records') and parsed_data.records:
            all_records = parsed_data.records
            print(f"✅ Successfully extracted {len(all_records)} records:")
            
            for i, record in enumerate(all_records, 1):
                print(f"--- Record {i} ---")
                print(f"No: {record.no}")
                print(f"Name: {record.provinsi}")
                print(f"Kabupaten/Kota: {record.kabupaten_kota}")
                print(f"Nama Kota: {record.nama_kota}")
                print(f"Singkatan Nama Kota: {record.singkatan_nama_kota}")
                print(f"Parent Subdivision: {record.parent_subdivision}")
                print()
        else:
            print("❌ No records found in parsed data")
            print("Parsed data:", parsed_data)
    else:
        print("❌ Failed to parse response")
        if chat_response:
            print("Response structure:", chat_response)
            
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

✅ Successfully extracted 72 records:
--- Record 1 ---
No: 66
Name: Sumatera Barat
Kabupaten/Kota: Kabupaten Dharmasraya
Nama Kota: Pulu Padung
Singkatan Nama Kota: PLJ
Parent Subdivision: ID-SB

--- Record 2 ---
No: 67
Name: Sumatera Barat
Kabupaten/Kota: Kota Solok
Nama Kota: Padang Aro
Singkatan Nama Kota: PDA
Parent Subdivision: ID-SB

--- Record 3 ---
No: 68
Name: Sumatera Barat
Kabupaten/Kota: Kabupaten Pasaman Barat
Nama Kota: Simpang Empat
Singkatan Nama Kota: SPE
Parent Subdivision: ID-SB

--- Record 4 ---
No: 69
Name: Sumatera Barat
Kabupaten/Kota: Kota Padang
Nama Kota: Padang
Singkatan Nama Kota: PAD
Parent Subdivision: ID-SB

--- Record 5 ---
No: 70
Name: Sumatera Barat
Kabupaten/Kota: Kota Solok
Nama Kota: Solok
Singkatan Nama Kota: SLK
Parent Subdivision: ID-SB

--- Record 6 ---
No: 71
Name: Sumatera Barat
Kabupaten/Kota: Kota Sawahlunto
Nama Kota: Sawahlunto
Singkatan Nama Kota: SWL
Parent Subdivision: ID-SB

--- Record 7 ---
No: 72
Name: Sumatera Barat
Kabupaten/Kota: K